In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import os
import math
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import (
    Dataset, DataLoader, WeightedRandomSampler, random_split
)
from tqdm import tqdm
from transformers.models.bert.modeling_bert import BertEncoder, BertConfig


print("Imports OK")
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")

INPUT_PATH = "/content/drive/MyDrive/DBAASP_Bioinformatics"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device  : {device}")

In [ ]:
"""
AMP Latent Diffusion — Phase 1 (unconditional) + Phase 2 (organism-conditioned)
"""

def timestep_embedding(timesteps, dim, max_period=10000):
    half  = dim // 2
    freqs = torch.exp(
        -math.log(max_period)
        * torch.arange(start=0, end=half, dtype=torch.float32)
        / half
    ).to(device=timesteps.device)
    args      = timesteps[:, None].float() * freqs[None]
    embedding = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)
    if dim % 2:
        embedding = torch.cat([embedding, torch.zeros_like(embedding[:, :1])], dim=-1)
    return embedding


class DModel(nn.Module):
    """
    Latent diffusion denoiser.
    num_class : int
        Phase 1: 2 (dummy, control_embedding present but never called).
        Phase 2: num_organisms + 1  (last index = unconditional / CFG null).
    max_length : int
        Always 1 — each peptide is a single 128-d latent vector.
    """
    def __init__(self, num_class=2, max_length=1):
        super().__init__()
        self.input_channels = 128
        self.model_channels = 127
        self.out_channels   = 128
        self.num_class      = num_class
        self.drop_out       = 0.1
        self.max_length     = max_length
        config = BertConfig()
        config.hidden_dropout_prob     = self.drop_out
        config.max_position_embeddings = self.max_length
        config.num_attention_heads     = 6
        config.num_hidden_layers       = 3
        config._attn_implementation    = "eager"
        self.control_embedding = nn.Embedding(self.num_class, config.hidden_size)
        self.time_embed = nn.Sequential(
            nn.Linear(self.model_channels, config.hidden_size),
            nn.SiLU(),
            nn.Linear(config.hidden_size, config.hidden_size),
        )
        self.input_up_proj = nn.Sequential(
            nn.Linear(self.input_channels, config.hidden_size),
            nn.Tanh(),
            nn.Linear(config.hidden_size, config.hidden_size),
        )
        self.input_transformers = BertEncoder(config)
        self.register_buffer(
            "position_ids",
            torch.arange(config.max_position_embeddings).expand((1, -1))
        )
        self.position_embeddings = nn.Embedding(
            config.max_position_embeddings, config.hidden_size
        )
        self.LayerNorm = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_eps)
        self.dropout   = nn.Dropout(config.hidden_dropout_prob)
        self.output_down_proj = nn.Sequential(
            nn.Linear(config.hidden_size, config.hidden_size),
            nn.Tanh(),
            nn.Linear(config.hidden_size, self.out_channels),
        )


    def forward(self, x, timesteps, control=None):
        """
        Parameters:
        x         : (B, 1, 128)
        timesteps : (B,)  long
        control   : (B,)  long or None
                    If None → unconditional forward pass (Phase 1 or CFG null).
        """
        emb = self.time_embed(timestep_embedding(timesteps, self.model_channels))
        if control is not None:
            emb = emb + self.control_embedding(control)

        emb_x      = self.input_up_proj(x)
        seq_length = x.size(1)
        pos_ids    = self.position_ids[:, :seq_length]

        emb_inputs = (
            self.position_embeddings(pos_ids)
            + emb_x
            + emb.unsqueeze(1).expand(-1, seq_length, -1)
        )
        emb_inputs = self.dropout(self.LayerNorm(emb_inputs))
        attn_mask = torch.ones(x.shape[0], seq_length, device=x.device)
        extended_mask = attn_mask[:, None, None, :]
        extended_mask = (1.0 - extended_mask) * torch.finfo(emb_inputs.dtype).min

        h = self.input_transformers(emb_inputs, attention_mask=extended_mask)[0]
        return self.output_down_proj(h).type(x.dtype)



def _extract_into_tensor(arr, timesteps, broadcast_shape):
    res = torch.from_numpy(arr).to(device=timesteps.device)[timesteps].float()
    while len(res.shape) < len(broadcast_shape):
        res = res[..., None]
    return res.expand(broadcast_shape)


def mean_flat(tensor):
    return tensor.mean(dim=list(range(1, len(tensor.shape))))


class MyDiffusion:
    def __init__(self, num_timesteps, betas):
        self.betas         = betas
        alphas             = 1.0 - betas
        self.num_timesteps = num_timesteps

        self.alphas_cumprod      = np.cumprod(alphas, axis=0)
        self.alphas_cumprod_prev = np.append(1.0, self.alphas_cumprod[:-1])

        self.sqrt_alphas_cumprod           = np.sqrt(self.alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = np.sqrt(1.0 - self.alphas_cumprod)

        self.posterior_variance = (
            betas * (1.0 - self.alphas_cumprod_prev) / (1.0 - self.alphas_cumprod)
        )
        self.posterior_log_variance_clipped = np.log(
            np.append(self.posterior_variance[1], self.posterior_variance[1:])
        )
        self.posterior_mean_coef1 = (
            betas * np.sqrt(self.alphas_cumprod_prev) / (1.0 - self.alphas_cumprod)
        )
        self.posterior_mean_coef2 = (
            (1.0 - self.alphas_cumprod_prev)
            * np.sqrt(alphas)
            / (1.0 - self.alphas_cumprod)
        )

    def q_sample(self, x_start, t, noise=None):
        if noise is None:
            noise = torch.randn_like(x_start)
        return (
            _extract_into_tensor(self.sqrt_alphas_cumprod, t, x_start.shape) * x_start
            + _extract_into_tensor(self.sqrt_one_minus_alphas_cumprod, t, x_start.shape)
            * noise
        )

    def _scale_timesteps(self, t):
        return t.float() * (1000.0 / self.num_timesteps)

    def get_x_start(self, x_start_mean, std):
        return x_start_mean + std * torch.randn_like(x_start_mean)

    def cal_loss(self, model, x_0, t, condition=None):
        """
        condition : (B,) long tensor or None
            For Phase 2 with CFG: pass the already-dropped labels (some entries
            will be num_organisms = unconditional index).
            For Phase 1: pass None.
        """
        std = _extract_into_tensor(
            self.sqrt_one_minus_alphas_cumprod,
            torch.tensor([0]).to(x_0.device),
            x_0.shape,
        )
        x_start      = self.get_x_start(x_0, std)
        noise        = torch.randn_like(x_0)
        x_t          = self.q_sample(x_start, t, noise=noise)
        model_output = model(x_t, self._scale_timesteps(t), condition)
        target       = x_start

        mse_loss = mean_flat((target - model_output) ** 2)
        t0_mask  = (t == 0)
        t0_loss  = mean_flat((x_0 - model_output) ** 2)
        return torch.where(t0_mask, t0_loss, mse_loss)


    def q_posterior_mean_variance(self, x_start, x_t, t):
        mean = (
            _extract_into_tensor(self.posterior_mean_coef1, t, x_t.shape) * x_start
            + _extract_into_tensor(self.posterior_mean_coef2, t, x_t.shape) * x_t
        )
        var     = _extract_into_tensor(self.posterior_variance,             t, x_t.shape)
        log_var = _extract_into_tensor(self.posterior_log_variance_clipped, t, x_t.shape)
        return mean, var, log_var


    def p_mean_variance(self, model, x, t, clip_denoised=True, cond=None):
        model_output   = model(x, self._scale_timesteps(t), cond)
        pred_xstart    = model_output.clamp(-1, 1) if clip_denoised else model_output
        model_mean, _, _ = self.q_posterior_mean_variance(pred_xstart, x, t)
        model_log_var  = _extract_into_tensor(
            self.posterior_log_variance_clipped, t, x.shape
        )
        return {
            "mean": model_mean,
            "log_variance": model_log_var,
            "pred_xstart": pred_xstart,
        }


    def p_sample(self, model, x, t, clip_denoised=False, cond=None):
        out          = self.p_mean_variance(model, x, t, clip_denoised, cond)
        noise        = torch.randn_like(x)
        nonzero_mask = (t != 0).float().view(-1, *([1] * (len(x.shape) - 1)))
        sample       = out["mean"] + nonzero_mask * torch.exp(0.5 * out["log_variance"]) * noise
        return {"sample": sample, "pred_xstart": out["pred_xstart"]}


    def p_sample_loop(self, model, shape, cond=None, device="cuda"):
        """Full reverse diffusion — used at inference."""
        img     = torch.randn(*shape, device=device)
        indices = list(range(self.num_timesteps))[::-1]
        for i in indices:
            t = torch.tensor([i] * shape[0], device=device)
            with torch.no_grad():
                out = self.p_sample(model, img, t, clip_denoised=False, cond=cond)
                img = out["sample"]
        return img


def betas_for_alpha_bar(num_diffusion_timesteps, alpha_bar, max_beta=0.999):
    betas = []
    for i in range(num_diffusion_timesteps):
        t1 = i / num_diffusion_timesteps
        t2 = (i + 1) / num_diffusion_timesteps
        betas.append(min(1 - alpha_bar(t2) / alpha_bar(t1), max_beta))
    return np.array(betas)


def create_gaussian_diffusion(steps=500):
    betas = betas_for_alpha_bar(steps, lambda t: 1 - np.sqrt(t + 0.0001))
    return MyDiffusion(num_timesteps=steps, betas=betas)


class UniformSampler:
    def __init__(self, num_timesteps=500):
        self._weights = np.ones([num_timesteps])

    def sample(self, batch_size, device):
        p          = self._weights / np.sum(self._weights)
        indices_np = np.random.choice(len(p), size=(batch_size,), p=p)
        indices    = torch.from_numpy(indices_np).long().to(device)
        weights_np = 1 / (len(p) * p[indices_np])
        weights    = torch.from_numpy(weights_np).float().to(device)
        return indices, weights


class LatentDataset_nocond(Dataset):
    """
    Phase 1 — loads a single .npy of shape (N, 128).
    encode_cond_latents.py saves raw mu (no ×25).
    """
    def __init__(self, npy_path):
        raw = np.load(npy_path).astype(np.float32)
        if raw.ndim == 3:
            raw = raw.squeeze(1)
        raw = raw * 25.0
        self.data = raw[:, np.newaxis, :]
        print(f"[nocond] {len(self.data)} samples  shape={self.data.shape[1:]}")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return torch.from_numpy(self.data[idx])


class LatentDataset_cond(Dataset):
    def __init__(self, latents_path, organisms_path):
        latents   = np.load(latents_path).astype(np.float32)
        organisms = np.load(organisms_path).astype(np.int64)

        if latents.ndim == 3:
            latents = latents.squeeze(1)

        assert len(latents) == len(organisms), (
            f"Length mismatch: {len(latents)} latents vs {len(organisms)} labels"
        )

        latents = latents * 25.0
        self.latents      = latents[:, np.newaxis, :]
        self.organisms    = organisms
        self.num_organisms = int(organisms.max()) + 1

        print(f"[cond] {len(self.latents)} samples  "
              f"organisms={self.num_organisms}")
        for i in range(self.num_organisms):
            count = (self.organisms == i).sum()
            print(f"  org {i}: {count} samples")

    def __len__(self):
        return len(self.latents)

    def __getitem__(self, idx):
        return (
            torch.from_numpy(self.latents[idx]),
            torch.tensor(self.organisms[idx], dtype=torch.long),
        )


def make_weighted_sampler(dataset):
    """
    Inverse-frequency WeightedRandomSampler so every organism gets
    equal expected gradient updates per epoch, regardless of class size.
    """
    organisms  = dataset.organisms
    num_orgs   = dataset.num_organisms
    counts     = np.bincount(organisms, minlength=num_orgs).astype(np.float32)
    class_weight = 1.0 / np.where(counts > 0, counts, 1.0)
    sample_weight = class_weight[organisms]
    sampler = WeightedRandomSampler(
        weights     = torch.from_numpy(sample_weight).float(),
        num_samples = len(sample_weight),
        replacement = True,
    )
    return sampler


print("Definitions loaded.")


P1 = dict(
    train_latents = f"{INPUT_PATH}/latents/train_latents.npy",
    val_latents   = f"{INPUT_PATH}/latents/val_latents.npy",
    save_path     = f"{INPUT_PATH}/checkpoints_diffusion/phase1",
    lr            = 1e-4,
    num_epochs    = 100,
    batch_size    = 512,
    num_steps     = 500,
    num_workers   = 2,
    save_every    = 10,
)

os.makedirs(P1["save_path"], exist_ok=True)
os.makedirs(f"{P1['save_path']}/model", exist_ok=True)

train_data_p1   = LatentDataset_nocond(P1["train_latents"])
val_data_p1     = LatentDataset_nocond(P1["val_latents"])
train_loader_p1 = DataLoader(
    train_data_p1,
    batch_size  = P1["batch_size"],
    shuffle     = True,
    num_workers = P1["num_workers"],
    pin_memory  = True,
    drop_last   = True,
)
val_loader_p1 = DataLoader(
    val_data_p1,
    batch_size  = P1["batch_size"] // 10,
    shuffle     = False,
    num_workers = P1["num_workers"],
    pin_memory  = True,
    drop_last   = False,
)
print(f"P1  Train={len(train_data_p1)}  Val={len(val_data_p1)}")
model_p1 = DModel(num_class=2, max_length=1).to(device)
diff_p1  = create_gaussian_diffusion(P1["num_steps"])
samp_p1  = UniformSampler(P1["num_steps"])
opt_p1   = AdamW(model_p1.parameters(), lr=P1["lr"])

log_path_p1 = f"{P1['save_path']}/train.log"
if not os.path.exists(log_path_p1):
    with open(log_path_p1, "w") as f:
        f.write("epoch,batch_idx,data_type,loss,run_time\n")

best_loss_p1 = float("inf")

for epoch in range(P1["num_epochs"]):
    model_p1.train()
    train_loss, n_train = 0.0, 0
    start    = time.time()
    log_rows = []

    for i, data in enumerate(tqdm(train_loader_p1, desc=f"[P1 Ep {epoch:3d}] train")):
        data = data.to(device)
        opt_p1.zero_grad()
        t, _ = samp_p1.sample(data.shape[0], device)
        loss = diff_p1.cal_loss(model_p1, data, t, condition=None).mean()
        loss.backward()
        opt_p1.step()
        train_loss += loss.item()
        n_train    += 1
        log_rows.append(
            f"{epoch},{i},train,{loss.item():.6f},{time.time()-start:.2f}"
        )

    train_loss /= n_train
    model_p1.eval()
    val_loss, n_val = 0.0, 0
    with torch.no_grad():
        for i, vdata in enumerate(tqdm(val_loader_p1, desc=f"[P1 Ep {epoch:3d}] val  ")):
            vdata = vdata.to(device)
            t, _  = samp_p1.sample(vdata.shape[0], device)
            loss  = diff_p1.cal_loss(model_p1, vdata, t, condition=None).mean()
            val_loss += loss.item()
            n_val    += 1
            log_rows.append(
                f"{epoch},{i},val,{loss.item():.6f},{time.time()-start:.2f}"
            )

    val_loss /= n_val

    with open(log_path_p1, "a") as f:
        f.write("\n".join(log_rows) + "\n")

    if val_loss < best_loss_p1:
        best_loss_p1 = val_loss
        torch.save(model_p1.state_dict(), f"{P1['save_path']}/best_model.pth")
        print(f"  ✓ Best Phase 1 model saved  val={val_loss:.5f}")

    if (epoch + 1) % P1["save_every"] == 0:
        ckpt = (f"{P1['save_path']}/model/"
                f"ep{epoch:04d}_tr{train_loss:.5f}_val{val_loss:.5f}.pth")
        torch.save(model_p1.state_dict(), ckpt)

    print(f"[P1] Ep {epoch:4d} | train {train_loss:.5f} | val {val_loss:.5f} "
          f"| {time.time()-start:.1f}s")

print("Phase 1 complete.")


P2 = dict(
    latents_path   = f"{INPUT_PATH}/latent_/cond_latents_new/cond_latents.npy",
    organisms_path = f"{INPUT_PATH}/latent_/cond_latents_new/cond_organisms.npy",
    phase1_ckpt    = f"{INPUT_PATH}/checkpoints_diffusion/phase1/best_model.pth",
    save_path      = f"{INPUT_PATH}/checkpoints_diffusion/phase2_final",
    lr             = 1e-4,
    num_epochs     = 300,
    batch_size     = 512,
    num_steps      = 500,
    num_workers    = 2,
    save_every     = 10,
    cfg_dropout    = 0.10,
)

os.makedirs(P2["save_path"], exist_ok=True)
os.makedirs(f"{P2['save_path']}/model", exist_ok=True)



all_data_p2   = LatentDataset_cond(P2["latents_path"], P2["organisms_path"])
num_organisms = all_data_p2.num_organisms
NULL_LABEL    = num_organisms

length     = len(all_data_p2)
train_size = int(0.8 * length)
val_size   = length - train_size
train_data_p2, val_data_p2 = random_split(
    all_data_p2, [train_size, val_size],
    generator=torch.Generator().manual_seed(42),
)

train_organisms = all_data_p2.organisms[list(train_data_p2.indices)]
counts     = np.bincount(train_organisms, minlength=num_organisms).astype(np.float32)
class_wt   = 1.0 / np.where(counts > 0, counts, 1.0)
sample_wts = class_wt[train_organisms]
weighted_sampler = WeightedRandomSampler(
    weights=torch.from_numpy(sample_wts).float(),
    num_samples=len(sample_wts),
    replacement=True,
)

train_loader_p2 = DataLoader(
    train_data_p2, batch_size=P2["batch_size"],
    sampler=weighted_sampler,
    num_workers=P2["num_workers"], pin_memory=True, drop_last=True,
)
val_loader_p2 = DataLoader(
    val_data_p2, batch_size=P2["batch_size"] // 10,
    shuffle=False,
    num_workers=P2["num_workers"], pin_memory=True, drop_last=False,
)
print(f"\nP2  Train={train_size}  Val={val_size}  "
      f"Organisms={num_organisms}  NullLabel={NULL_LABEL}")



model_p2 = DModel(num_class=num_organisms + 1, max_length=1).to(device)

if os.path.exists(P2["phase1_ckpt"]):
    p1_ckpt = torch.load(P2["phase1_ckpt"], map_location=device)
    if isinstance(p1_ckpt, dict) and "model_state" in p1_ckpt:
        p1_ckpt = p1_ckpt["model_state"]
    p2_state = model_p2.state_dict()
    to_load  = {k: v for k, v in p1_ckpt.items()
                if k in p2_state and v.shape == p2_state[k].shape}
    skipped  = [k for k in p1_ckpt if k not in to_load]
    model_p2.load_state_dict(to_load, strict=False)
    print(f"Loaded {len(to_load)} tensors from Phase 1. Skipped: {skipped}")
else:
    print("WARNING: Phase 1 checkpoint not found — training from scratch.")

diff_p2 = create_gaussian_diffusion(P2["num_steps"])
samp_p2 = UniformSampler(P2["num_steps"])
opt_p2  = AdamW(model_p2.parameters(), lr=P2["lr"])

log_path_p2 = f"{P2['save_path']}/train.log"
if not os.path.exists(log_path_p2):
    with open(log_path_p2, "w") as f:
        f.write("epoch,batch_idx,data_type,loss,run_time\n")


# training loop
best_loss_p2 = float("inf")
cfg_dropout  = P2["cfg_dropout"]
print("\nStarting Phase 2 training...")

for epoch in range(P2["num_epochs"]):
    model_p2.train()
    train_loss, n_train = 0.0, 0
    start    = time.time()
    log_rows = []

    for i, (data, organism_ids) in enumerate(
        tqdm(train_loader_p2, desc=f"[P2 Ep {epoch:3d}] train")
    ):
        data         = data.to(device)
        organism_ids = organism_ids.to(device)
        drop_mask    = torch.rand(organism_ids.shape[0], device=device) < cfg_dropout
        organism_ids = torch.where(
            drop_mask,
            torch.full_like(organism_ids, NULL_LABEL),
            organism_ids,
        )
        opt_p2.zero_grad()
        t, _ = samp_p2.sample(data.shape[0], device)
        loss  = diff_p2.cal_loss(model_p2, data, t, condition=organism_ids).mean()
        loss.backward()
        opt_p2.step()
        train_loss += loss.item()
        n_train    += 1
        log_rows.append(f"{epoch},{i},train,{loss.item():.6f},{time.time()-start:.2f}")

    train_loss /= n_train

    model_p2.eval()
    val_loss, n_val = 0.0, 0
    with torch.no_grad():
        for i, (vdata, organism_ids) in enumerate(
            tqdm(val_loader_p2, desc=f"[P2 Ep {epoch:3d}] val  ")
        ):
            vdata        = vdata.to(device)
            organism_ids = organism_ids.to(device)
            t, _  = samp_p2.sample(vdata.shape[0], device)
            loss  = diff_p2.cal_loss(model_p2, vdata, t, condition=organism_ids).mean()
            val_loss += loss.item()
            n_val    += 1
            log_rows.append(f"{epoch},{i},val,{loss.item():.6f},{time.time()-start:.2f}")

    val_loss /= n_val

    with open(log_path_p2, "a") as f:
        f.write("\n".join(log_rows) + "\n")

    if val_loss < best_loss_p2:
        best_loss_p2 = val_loss
        torch.save(
            {
                "model_state":   model_p2.state_dict(),
                "num_organisms": num_organisms,
                "null_label":    NULL_LABEL,
                "num_steps":     P2["num_steps"],
            },
            f"{P2['save_path']}/best_model.pth",
        )
        print(f"  ✓ Best model saved  val={val_loss:.5f}")

    if (epoch + 1) % P2["save_every"] == 0:
        torch.save(
            {
                "model_state":   model_p2.state_dict(),
                "num_organisms": num_organisms,
                "null_label":    NULL_LABEL,
                "num_steps":     P2["num_steps"],
            },
            f"{P2['save_path']}/model/"
            f"ep{epoch:04d}_tr{train_loss:.5f}_val{val_loss:.5f}.pth",
        )

    print(f"[P2] Ep {epoch:4d} | train {train_loss:.5f} | val {val_loss:.5f} "
          f"| {time.time()-start:.1f}s")

print("Phase 2 training complete.")

Mounted at /content/drive
Imports OK
PyTorch : 2.11.0+cu128
CUDA    : True
Device  : cuda
Definitions loaded.
[nocond] 456400 samples  shape=(1, 128)
[nocond] 23958 samples  shape=(1, 128)
P1  Train=456400  Val=23958


[P1 Ep   0] val  : 100%|██████████| 470/470 [00:02<00:00, 161.22it/s]


  ✓ Best Phase 1 model saved  val=219.42755
[P1] Ep    0 | train 281.34882 | val 219.42755 | 31.7s


[P1 Ep   1] val  : 100%|██████████| 470/470 [00:03<00:00, 135.17it/s]


  ✓ Best Phase 1 model saved  val=143.16118
[P1] Ep    1 | train 179.60619 | val 143.16118 | 31.6s


...

[P2] Ep  297 | train 4.04338 | val 7.22808 | 1.4s


[P2 Ep 298] val  : 100%|██████████| 36/36 [00:00<00:00, 75.44it/s]


[P2] Ep  298 | train 4.52995 | val 7.01980 | 1.2s


[P2 Ep 299] val  : 100%|██████████| 36/36 [00:00<00:00, 73.72it/s]


[P2] Ep  299 | train 4.09149 | val 7.03388 | 1.4s
Phase 2 training complete.
